## Cargar archivos

In [68]:
import os

# En local tenemos los archivos en una carpeta llamada genes
# En colab los tenemos en el root
carpeta = "genes" if os.path.isdir("genes") else ""

def leer_secuencia(ruta):
    with open(ruta, encoding="utf-8") as f:
        next(f) # salta la primera línea (encabezado)
        secuencia = ""
        for linea in f:
            secuencia += linea.strip() # quita el salto de línea
        return secuencia


gen_M = leer_secuencia(os.path.join(carpeta, "gen-M.txt"))
gen_S = leer_secuencia(os.path.join(carpeta, "gen-S.txt"))
gen_ORF1AB = leer_secuencia(os.path.join(carpeta, "gen-ORF1AB.txt"))
SARS_COV_2_MN908947_3 = leer_secuencia(os.path.join(carpeta, "SARS-COV-2-MN908947.3.txt"))
SARS_COV_2_MT106054_1 = leer_secuencia(os.path.join(carpeta, "SARS-COV-2-MT106054.1.txt"))

## Algoritmos utilizados

#### KMP CON LPS PARA STRING MATCHING

In [69]:
def construir_lps(patron):
    lps = [0] * len(patron)
    longitud = 0
    i = 1
    while i < len(patron):
        if patron[i] == patron[longitud]:
            longitud += 1
            lps[i] = longitud
            i += 1
        elif longitud:
            longitud = lps[longitud - 1]
        else:
            lps[i] = 0
            i += 1
    return lps


def kmp_buscar(texto, patron):
    """Devuelve los índices donde aparece patron en texto."""
    n, m = len(texto), len(patron)
    if m == 0:
        return []
    lps = construir_lps(patron)
    indices = []
    i = j = 0 # índices para iterar en texto y patron
    while i < n:
        if texto[i] == patron[j]:
            i += 1
            j += 1
        else:
            if j > 0:
                j = lps[j - 1]
            else:
                i += 1

        if j == m: # patron existe a partir de i - m
            indices.append(i - m)
            j = lps[j - 1]
    return indices

#### MANACHER PARA ENCONTRAR PALINDROMOS

In [70]:
# Complemento de cada base, '#' es su propio complemento
COMPLEMENTO = {
    "A": "T",
    "T": "A",
    "C": "G",
    "G": "C",
    "#": "#"
}


def manacher(texto, biologico=False):
    """Si biologico=True se usa la definición biológica de palíndromo, en
    donde un lado debe ser el complemento del otro."""
    t = "^#" + "#".join(texto) + "#$"
    e = len(t)
    P = [0] * e
    centro = limite = 0

    for i in range(1, e - 1):
        # Ninguna base es su propio complemento: solo hay centros entre bases
        if biologico and t[i] != "#":
            continue

        if i < limite:
            simetrica = 2 * centro - i
            P[i] = min(limite - i, P[simetrica])

        gap = P[i] + 1
        while t[i - gap] == (COMPLEMENTO.get(t[i + gap]) if biologico else t[i + gap]):
            P[i] += 1
            gap += 1

        if i + P[i] > limite:
            limite = i + P[i]
            centro = i
    return P


def palindromo_mas_largo(texto, biologico=False):
    """Devuelve (inicio, longitud) del palíndromo más largo de texto."""
    P = manacher(texto, biologico)
    longitud, centro = max((r, i) for i, r in enumerate(P))
    inicio = (centro - longitud) // 2
    return inicio, longitud

## Punto 1
#### Encontrar los índices de aparición de cada uno de los tres genes (gen M, S y ORF1AB) en la secuencia del virus SARS-COV-2-MN908947.3.txt.
Por cada gen muestra su nombre, índices de aparición en la secuencia del virus y primeros 12 caracteres.

In [71]:
genes = {
    "M": gen_M,
    "S": gen_S,
    "ORF1AB": gen_ORF1AB
}

for nombre, gen in genes.items():
    indices = kmp_buscar(SARS_COV_2_MN908947_3, gen)
    print(f"Gen {nombre}")
    print(f"  Índices de aparición: {indices}")
    print(f"  Primeros 12 caracteres: {gen[:12]}")

Gen M
  Índices de aparición: [26522]
  Primeros 12 caracteres: ATGGCAGATTCC
Gen S
  Índices de aparición: [21562]
  Primeros 12 caracteres: ATGTTTGTTTTT
Gen ORF1AB
  Índices de aparición: [265]
  Primeros 12 caracteres: ATGGAGAGCCTT


## Punto 2
#### Encontrar el palíndromo mas largo en cada uno de los tres genes (gen M, S y ORF1AB). Los palíndromos son importantes, porque son regiones propensas a mutaciones en un gen.

Por cada gen, muestra la longitud del palíndromo mas largo y guárdalo en un archivo.

In [72]:
with open("palindromos.txt", "w", encoding="utf-8") as salida:
    for nombre, gen in genes.items():
        inicio, longitud = palindromo_mas_largo(gen)
        palindromo = gen[inicio:inicio + longitud]
        print(f"Gen {nombre}: longitud {longitud} (índice {inicio}) -> {palindromo}")
        salida.write(f"Gen {nombre}\nLongitud: {longitud}\nÍndice: {inicio}\nPalíndromo: {palindromo}\n\n")

Gen M: longitud 11 (índice 493) -> CTAAAGAAATC
Gen S: longitud 15 (índice 2386) -> ATTTTGGTGGTTTTA
Gen ORF1AB: longitud 20 (índice 9711) -> CTCAATGACTTCAGTAACTC


#### Palíndromo biológico
Un palíndromo en biología es una secuencia de ADN igual a su complemento en reverso. El complemento se consigue cambiando A <-> T y C <-> G. Por ejemplo para GAATTC, su complemento es CTTAAG. Luego, si le hacemos reverse obtenemos la original.

In [73]:
for nombre, gen in genes.items():
    inicio, longitud = palindromo_mas_largo(gen, biologico=True)
    palindromo = gen[inicio:inicio + longitud]
    print(f"Gen {nombre}: longitud {longitud} (índice {inicio}) -> {palindromo}")

Gen M: longitud 10 (índice 310) -> CGCGTACGCG
Gen S: longitud 16 (índice 1338) -> GGTAATTATAATTACC
Gen ORF1AB: longitud 20 (índice 5479) -> ACACTGGTAATTACCAGTGT


## Punto 3
#### Dado el archivo de las proteínas, el cual contiene sus secuencias de aminoácidos, encuentra en cuales secciones del virus se produce cada proteína.

Muestra el nombre de la proteína y los índices del genoma del virus que la producen. Muestra sus primeros 4 aminoácidos, y codones asociados del virus.

In [74]:
# Tabla del código genético estándar
CODONES = {
    'ATA': 'I', 'ATC': 'I', 'ATT': 'I', 'ATG': 'M',
    'ACA': 'T', 'ACC': 'T', 'ACG': 'T', 'ACT': 'T',
    'AAC': 'N', 'AAT': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGC': 'S', 'AGT': 'S', 'AGA': 'R', 'AGG': 'R',
    'CTA': 'L', 'CTC': 'L', 'CTG': 'L', 'CTT': 'L',
    'CCA': 'P', 'CCC': 'P', 'CCG': 'P', 'CCT': 'P',
    'CAC': 'H', 'CAT': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGA': 'R', 'CGC': 'R', 'CGG': 'R', 'CGT': 'R',
    'GTA': 'V', 'GTC': 'V', 'GTG': 'V', 'GTT': 'V',
    'GCA': 'A', 'GCC': 'A', 'GCG': 'A', 'GCT': 'A',
    'GAC': 'D', 'GAT': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGA': 'G', 'GGC': 'G', 'GGG': 'G', 'GGT': 'G',
    'TCA': 'S', 'TCC': 'S', 'TCG': 'S', 'TCT': 'S',
    'TTC': 'F', 'TTT': 'F', 'TTA': 'L', 'TTG': 'L',
    'TAC': 'Y', 'TAT': 'Y', 'TAA': '_', 'TAG': '_',
    'TGC': 'C', 'TGT': 'C', 'TGA': '_', 'TGG': 'W',
}


def leer_fasta(ruta):
    """Devuelve {id: secuencia} de un archivo FASTA con varias entradas."""
    secuencias = {}
    with open(ruta, encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()
            if linea.startswith(">"):
                actual = linea[1:].split()[0]
                secuencias[actual] = []
            elif linea:
                secuencias[actual].append(linea)
    return {nombre: "".join(partes) for nombre, partes in secuencias.items()}


def traducir(nucleotidos, marco=0):
    """Traduce nucleotidos a aminoácidos."""
    proteina = ""
    for i in range(marco, len(nucleotidos) - 2, 3):
        codon = nucleotidos[i:i + 3]
        proteina += CODONES[codon]
    return proteina

In [75]:
proteinas = leer_fasta(os.path.join(carpeta, "seq-proteins.txt"))
genoma = SARS_COV_2_MN908947_3
traducciones = [traducir(genoma, marco) for marco in range(3)]


def buscar_proteina(secuencia):
    """Posiciones del genoma donde se codifica secuencia.
    Depende de donde empecemos a contar (0,1 o 2) podemos obtener
    resultados distintos."""
    return [
        marco + 3 * i
        for marco in range(3)
        for i in kmp_buscar(traducciones[marco], secuencia)
    ]


def tramos_proteina(secuencia):
    inicios = buscar_proteina(secuencia)
    if inicios: # En este caso se encontro la proteína completa sin saltos
        return [(0, inicios[0], len(secuencia))]

    # Si no aparece completa, buscamos el prefijo más largo que sí aparece
    # Para esto, usamos binary search sobre la longitud del prefijo de la
    # proteína
    bajo, alto = 1, len(secuencia) - 1
    while bajo < alto:
        medio = (bajo + alto + 1) // 2
        if buscar_proteina(secuencia[:medio]):
            bajo = medio
        else:
            alto = medio - 1

    inicios_prefijo = buscar_proteina(secuencia[:bajo])
    inicios_resto = buscar_proteina(secuencia[bajo:])
    # Si no encontramos el prefijo o el sufijo, no existe la proteína
    if not inicios_prefijo or not inicios_resto:
        return []
    return [(0, inicios_prefijo[0], bajo), (bajo, inicios_resto[0], len(secuencia) - bajo)]


for nombre, secuencia in proteinas.items():
    tramos = tramos_proteina(secuencia)
    print(f"Proteína {nombre} ({len(secuencia)} aa)")
    if not tramos:
        print("  No encontrada en el genoma")
        continue
    for _, inicio, n_aa in tramos:
        fin = inicio + 3 * n_aa - 1
        print(f"  Índices del genoma: {inicio}-{fin}")
    inicio = tramos[0][1]
    codones = [genoma[inicio + 3 * k: inicio + 3 * k + 3] for k in range(4)]
    print(f"  Primeros 4 aminoácidos: {secuencia[:4]}")
    print(f"  Codones: {' '.join(codones)}")
    if len(tramos) > 1:
        print("  (cambio de marco de lectura entre los dos tramos)")

Proteína QHD43415_1 (180 aa)
  Índices del genoma: 265-804
  Primeros 4 aminoácidos: MESL
  Codones: ATG GAG AGC CTT
Proteína QHD43415_2 (638 aa)
  Índices del genoma: 805-2718
  Primeros 4 aminoácidos: AYTR
  Codones: GCA TAC ACT CGC
Proteína QHD43415_3 (1945 aa)
  Índices del genoma: 2719-8553
  Primeros 4 aminoácidos: APTK
  Codones: GCA CCA ACA AAG
Proteína QHD43415_4 (500 aa)
  Índices del genoma: 8554-10053
  Primeros 4 aminoácidos: KIVN
  Codones: AAA ATT GTT AAT
Proteína QHD43415_5 (306 aa)
  Índices del genoma: 10054-10971
  Primeros 4 aminoácidos: SGFR
  Codones: AGT GGT TTT AGA
Proteína QHD43415_6 (290 aa)
  Índices del genoma: 10972-11841
  Primeros 4 aminoácidos: SAVK
  Codones: AGT GCA GTG AAA
Proteína QHD43415_7 (83 aa)
  Índices del genoma: 11842-12090
  Primeros 4 aminoácidos: SKMS
  Codones: TCT AAA ATG TCA
Proteína QHD43415_8 (198 aa)
  Índices del genoma: 12091-12684
  Primeros 4 aminoácidos: AIAS
  Codones: GCT ATA GCC TCA
Proteína QHD43415_9 (113 aa)
  Índices del

## Punto 4

#### Compara las versiones del genoma del virus de Wuhan, 2019 vs Texas, 2020. Determina donde  difieren, y si tales diferencias resultan en aminoácidos diferentes.

Lista los índices donde encuentres diferencias, muestra los codones afectados por tal diferencia y el aminoácido producido por cada versión.
¿Cual es la cadena mas larga sin diferencias entre ambos genomas?

In [76]:
wuhan, texas = SARS_COV_2_MN908947_3, SARS_COV_2_MT106054_1

# Comparamos solo las bases comunes: el genoma más largo solo tiene de más la cola poli-A
n = min(len(wuhan), len(texas))
diferencias = [i for i in range(n) if wuhan[i] != texas[i]]

In [77]:
# Regiones del genoma que codifican cada proteína (según el punto 3)
regiones = []
for nombre, secuencia in proteinas.items():
    for _, inicio, n_aa in tramos_proteina(secuencia):
        fin = inicio + 3 * n_aa  # exclusivo
        regiones.append((inicio, fin))


def inicio_region(indice):
    """Devuelve el inicio de la región codificante que contiene indice, o None si no hay."""
    for inicio, fin in regiones:
        if inicio <= indice < fin:
            return inicio
    return None


for i in diferencias:
    print(f"Índice {i}: Wuhan {wuhan[i]} -> Texas {texas[i]}")
    inicio = inicio_region(i)
    if inicio is None:
        print("  Fuera de las regiones codificantes: no afecta aminoácidos")
        continue

    inicio_codon = inicio + 3 * ((i - inicio) // 3)
    codon_wuhan = wuhan[inicio_codon:inicio_codon + 3]
    codon_texas = texas[inicio_codon:inicio_codon + 3]
    aa_wuhan = CODONES[codon_wuhan]
    aa_texas = CODONES[codon_texas]

    if aa_wuhan == aa_texas:
        efecto = "mismo aminoácido"
    else:
        efecto = "aminoácido diferente"
    print(f"  Wuhan: {codon_wuhan} -> {aa_wuhan} | Texas: {codon_texas} -> {aa_texas}  ({efecto})")

Índice 8781: Wuhan C -> Texas T
  Wuhan: AGC -> S | Texas: AGT -> S  (mismo aminoácido)
Índice 18602: Wuhan T -> Texas C
  Wuhan: CAT -> H | Texas: CAC -> H  (mismo aminoácido)
Índice 18974: Wuhan T -> Texas A
  Wuhan: GTT -> V | Texas: GTA -> V  (mismo aminoácido)
Índice 19174: Wuhan A -> Texas C
  Wuhan: GAT -> D | Texas: GCT -> A  (aminoácido diferente)
Índice 27924: Wuhan C -> Texas T
  Wuhan: ACA -> T | Texas: ATA -> I  (aminoácido diferente)
Índice 28143: Wuhan T -> Texas C
  Wuhan: TTA -> L | Texas: TCA -> S  (aminoácido diferente)
Índice 29094: Wuhan C -> Texas T
  Wuhan: TTC -> F | Texas: TTT -> F  (mismo aminoácido)


In [78]:
# Cadena más larga sin diferencias
limites = [-1] + diferencias + [n]
mejor_longitud = mejor_inicio = 0
for anterior, siguiente in zip(limites, limites[1:]):
    longitud = siguiente - anterior - 1
    if longitud > mejor_longitud:
        mejor_longitud = longitud
        mejor_inicio = anterior + 1

mejor_fin = mejor_inicio + mejor_longitud - 1
print(f"Cadena más larga sin diferencias: {mejor_longitud} bases")
print(f"  Índices (iguales en ambos genomas): {mejor_inicio}-{mejor_fin}")

Cadena más larga sin diferencias: 9820 bases
  Índices (iguales en ambos genomas): 8782-18601
